# hw 4
1. Загрузка и разметка данных Collection5
2. Разбиение на обучающую и тестовую выборки
3. Подготовка данных для NER: токенизация и выравнивание BIO-меток
4. Финетюнинг rubert-tiny2 для NER
5. Этап MLM и повторная дообучение для улучшения
6. Дообучение с синтетикой

In [1]:
import random

import numpy as np
import pandas as pd
import torch
import evaluate
from evaluate import load

from corus import load_ne5
import corus.sources.ne5 as ne5
from razdel import tokenize
from sklearn.model_selection import train_test_split
from datasets import Dataset, concatenate_datasets
from typing import List, Dict, Tuple
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)


# --- Константы и параметры ---
SEED = 43
MODEL_CHECKPOINT = 'cointegrated/rubert-tiny2'
block_size = 512 # длина блока для MLM: кратная архитектуре BERT

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [2]:
# читаем raw-текст NE5 в utf-8
def load_text_utf8(path):
    """Чтение текста в кодировке UTF-8 для ne5."""
    with open(path, encoding='utf-8') as f:
        return f.read()

ne5.load_text = load_text_utf8
corpus = list(load_ne5('data/Collection5/'))

In [3]:
corpus[0].text

'Россия рассчитывает на конструктивное воздействие США на Грузию\n\n04/08/2008 12:08\n\nМОСКВА, 4 авг - РИА Новости. Россия рассчитывает, что США воздействуют на Тбилиси в связи с обострением ситуации в зоне грузино-осетинского конфликта. Об этом статс-секретарь - заместитель министра иностранных дел России Григорий Карасин заявил в телефонном разговоре с заместителем госсекретаря США Дэниэлом Фридом.\n\n"С российской стороны выражена глубокая озабоченность в связи с новым витком напряженности вокруг Южной Осетии, противозаконными действиями грузинской стороны по наращиванию своих вооруженных сил в регионе, бесконтрольным строительством фортификационных сооружений", - говорится в сообщении.\n\n"Россия уже призвала Тбилиси к ответственной линии и рассчитывает также на конструктивное воздействие со стороны Вашингтона", - сообщил МИД России. '

In [4]:
#Формирование примеров, в формате {'text':..., 'entities':[...]}
examples = [
    {
        'text': doc.text,
        'entities': [
            {'start': s.start, 'end': s.stop, 'type': s.type}
            for s in doc.spans
        ],
    }
    for doc in corpus
]

In [5]:
#разбиение на train/test
df_train, df_test = train_test_split(
    examples,
    test_size=0.2,
    random_state=SEED,
)

In [6]:
#Инициализация токенизатора и меток
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
# B/I по каждому типу
entity_types = sorted({ent['type'] for ex in examples for ent in ex['entities']})
label2id = {'O': 0}
for idx, et in enumerate(entity_types, start=1):
    label2id[f'B-{et}'] = idx * 2 - 1
    label2id[f'I-{et}'] = idx * 2
id2label = {v: k for k, v in label2id.items()}

In [7]:
def prepare_for_ner(example):
    """Токенизация текста и выравнивание BIO-меток на уровне токенов."""
    tokens = list(tokenize(example['text']))
    words = [t.text for t in tokens]
    tags = ['O'] * len(words)

    for ent in example['entities']:
        indices = [i for i, tok in enumerate(tokens) if tok.start < ent['end'] and tok.stop > ent['start']]
        for pos, i in enumerate(indices):
            prefix = 'B' if pos == 0 else 'I'
            tags[i] = f'{prefix}-{ent["type"]}'

    enc = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
    )
    word_ids = enc.word_ids()
    enc['labels'] = [
        label2id[tags[w]] if w is not None else -100
        for w in word_ids
    ]
    return enc

In [8]:
# Преобразуем в HF Dataset
df_train = Dataset.from_list(df_train).map(
    prepare_for_ner,
    remove_columns=['text', 'entities'],
)
df_test = Dataset.from_list(df_test).map(
    prepare_for_ner,
    remove_columns=['text', 'entities'],
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## NER Finetuning

In [9]:
ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
data_collator = DataCollatorForTokenClassification(tokenizer)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
metric = evaluate.load("seqeval")
label_names = list(label2id.keys()) 

def compute_metrics(eval_preds: Tuple[np.ndarray, np.ndarray]) -> Dict[str, float]:
    """
    Вычисляет precision, recall, f1 и accuracy для NER,
    игнорируя токены с меткой -100.
    """
    logits, label_ids = eval_preds
    pred_ids = np.argmax(logits, axis=-1)

    true_seqs: List[List[str]] = []
    pred_seqs: List[List[str]] = []

    for true_seq, pred_seq in zip(label_ids, pred_ids):
        # фильтрация игнорируемых токенов
        pairs = [(p, t) for p, t in zip(pred_seq, true_seq) if t != -100]
        true_seqs.append([label_names[t] for _, t in pairs])
        pred_seqs.append([label_names[p] for p, _ in pairs])

    results = metric.compute(
        predictions=pred_seqs,
        references=true_seqs,
        zero_division=0
    )

    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

In [11]:
# ## NER Finetuning (AdamW, lr=2e-5, batch=16, epochs=10)
# стандартный тонкий тюнинг задачи token-classification
args = TrainingArguments(
    output_dir='ner',
    eval_strategy='epoch', # валидация после каждой эпохи для контроля
    learning_rate=2e-5, # стандартный LR для fine-tune
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10, # по нескольким прогонам эпох достаточно для сходимости
    weight_decay=0.01,
    seed=SEED,
)

trainer = Trainer(
    model=ner_model,
    args=args,
    train_dataset=df_train,
    eval_dataset=df_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

trainer.evaluate()

{'eval_loss': 2.4627349376678467,
 'eval_model_preparation_time': 0.0,
 'eval_precision': 0.013558866962008323,
 'eval_recall': 0.08984623204981573,
 'eval_f1': 0.023561954275811505,
 'eval_accuracy': 0.07905649038461539,
 'eval_runtime': 2.4921,
 'eval_samples_per_second': 80.252,
 'eval_steps_per_second': 5.216}

In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,No log,1.046721,0.000000,0.170732,0.000890,0.001770,0.731656
2,No log,0.861028,0.000000,0.161666,0.095184,0.119821,0.765700
3,No log,0.766952,0.000000,0.169833,0.112467,0.135321,0.778921
4,No log,0.709063,0.000000,0.178891,0.134833,0.153768,0.788612
5,No log,0.670295,0.000000,0.203771,0.156564,0.177075,0.795959
6,No log,0.643582,0.000000,0.221986,0.185792,0.202283,0.802704
7,No log,0.627123,0.000000,0.233499,0.209048,0.220598,0.807197
8,No log,0.613018,0.000000,0.248803,0.217944,0.232353,0.812831
9,No log,0.605922,0.000000,0.254941,0.226204,0.239714,0.814709
10,0.834500,0.603481,0.000000,0.257742,0.229508,0.242807,0.815385


TrainOutput(global_step=500, training_loss=0.8345173950195313, metrics={'train_runtime': 862.1403, 'train_samples_per_second': 9.279, 'train_steps_per_second': 0.58, 'total_flos': 88880837762016.0, 'train_loss': 0.8345173950195313, 'epoch': 10.0})

In [13]:
trainer.evaluate()

{'eval_loss': 0.6034805774688721,
 'eval_model_preparation_time': 0.0,
 'eval_precision': 0.25774225774225773,
 'eval_recall': 0.22950819672131148,
 'eval_f1': 0.2428072062382361,
 'eval_accuracy': 0.8153846153846154,
 'eval_runtime': 10.4174,
 'eval_samples_per_second': 19.199,
 'eval_steps_per_second': 1.248,
 'epoch': 10.0}

In [14]:
trainer.save_model("./ner_finetuned_model")

# 2. MLM дообучение

In [15]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [16]:
df_train_mlm = df_train.map(group_texts, batched=True,)
df_test_mlm = df_test.map(group_texts, batched=True,)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [17]:
tokenizer = tokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForMaskedLM.from_pretrained(MODEL_CHECKPOINT)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm_probability=0.15)

In [18]:
mlm_args = TrainingArguments(
    output_dir='mlm_out',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=mlm_args,
    train_dataset=df_train_mlm,
    eval_dataset=df_train_mlm,
    data_collator=data_collator,
)


trainer.evaluate()

{'eval_loss': 3.123777151107788,
 'eval_model_preparation_time': 0.0,
 'eval_runtime': 26.8475,
 'eval_samples_per_second': 17.879,
 'eval_steps_per_second': 1.117}

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,3.069488,0.000000
2,No log,3.018744,0.000000
3,No log,2.994252,0.000000
4,No log,2.954250,0.000000
5,No log,2.925057,0.000000
6,No log,2.937265,0.000000
7,No log,2.940333,0.000000
8,No log,2.907559,0.000000
9,No log,2.888377,0.000000
10,No log,2.897658,0.000000


TrainOutput(global_step=300, training_loss=3.294347330729167, metrics={'train_runtime': 119.3098, 'train_samples_per_second': 40.231, 'train_steps_per_second': 2.514, 'total_flos': 36632258150400.0, 'train_loss': 3.294347330729167, 'epoch': 10.0})

In [20]:
trainer.evaluate()

{'eval_loss': 2.9100916385650635,
 'eval_model_preparation_time': 0.0,
 'eval_runtime': 3.0506,
 'eval_samples_per_second': 157.346,
 'eval_steps_per_second': 9.834,
 'epoch': 10.0}

In [21]:
trainer.save_model("./ner_pretrained_mlm_model")

In [22]:
# Перезагрузка модели для NER после MLM
model = AutoModelForTokenClassification.from_pretrained(
    "./ner_pretrained_mlm_model",
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
data_collator = DataCollatorForTokenClassification(tokenizer)
tokenizer = tokenizer.from_pretrained(MODEL_CHECKPOINT)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ./ner_pretrained_mlm_model and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
# используем дообученную MLM-модель как основу для NER
args = TrainingArguments(
    output_dir='ner',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    seed=SEED,
)

In [24]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df_train,
    eval_dataset=df_test,
    data_collator=data_collator,
)

compute_metrics((trainer.predict(df_test).predictions, df_test['labels']))

{'precision': 0.012483827768572531,
 'recall': 0.06989452281103063,
 'f1': 0.021183992604860764,
 'accuracy': 0.08635817307692308}

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time
1,No log,1.050251,0.000000
2,No log,0.870966,0.000000
3,No log,0.772973,0.000000
4,No log,0.711156,0.000000
5,No log,0.672202,0.000000
6,No log,0.644765,0.000000
7,No log,0.626836,0.000000
8,No log,0.612780,0.000000
9,No log,0.605382,0.000000
10,0.838900,0.603095,0.000000


TrainOutput(global_step=500, training_loss=0.8389060668945313, metrics={'train_runtime': 891.9529, 'train_samples_per_second': 8.969, 'train_steps_per_second': 0.561, 'total_flos': 88880837762016.0, 'train_loss': 0.8389060668945313, 'epoch': 10.0})

In [26]:
compute_metrics((trainer.predict(df_test).predictions, df_test['labels']))

{'precision': 0.2557758031442242,
 'recall': 0.23776845850806963,
 'f1': 0.24644362486828242,
 'accuracy': 0.8137920673076923}

In [27]:
trainer.save_model("./ner_finetuned_mlm_model")

# 3. Использование синтетики для дообучения

In [28]:
#Синтетическая разметка (10k примеров lenta) 
lenta_data = pd.read_parquet('lenta_data.parquet')
lenta_data = lenta_data.rename(columns={'text': 'words', 'annotation': 'bio_labels'})

In [29]:
lenta_data

,words,bio_labels
0,"[В, афганской, провинции, Нангархар, ,, находя...","[O, O, O, B-LOC, O, O, O, O, O, O, O, O, O, O,..."
1,"[Отток, капитала, из, России, по, итогам, перв...","[O, O, O, B-LOC, O, O, O, O, O, O, O, O, O, O,..."
2,"[Эдриану, Броуди, предложили, роль, в, триллер...","[B-PER, I-PER, O, O, O, O, B-PER, I-PER, O, O,..."
3,"[Во, вторник, в, Красноярск, прибыла, съемочна...","[O, O, O, B-LOC, O, O, O, O, O, O, O, O, O, O,..."
4,"[В, четверг, три, жителя, Тегерана, были, подв...","[O, O, O, O, B-LOC, O, O, O, O, O, O, O, O, O,..."
...,...,...
9995,"[Глава, российской, Ассоциации, интернет, -, и...","[O, O, B-ORG, I-ORG, I-ORG, I-ORG, B-PER, I-PE..."
9996,"[Кипрская, компания, Xomeric, ,, которая, связ...","[O, O, B-ORG, O, O, O, O, O, O, B-ORG, I-ORG, ..."
9997,"[Европейские, астрономы, обнаружили, суперземл...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
9998,"[Правительство, Мьянмы, ,, правоохранительные,...","[O, B-LOC, O, O, O, O, O, O, O, O, O, O, O, O,..."


In [30]:
def prepare_synth(example):
    """Токенизация слов и выравнивание списка BIO-меток на уровне токенов."""
    enc = tokenizer(
        example['words'],
        is_split_into_words=True,
        truncation=True
    )
    word_ids = enc.word_ids()
    enc['labels'] = [
        label2id[example['bio_labels'][w]] if w is not None else -100
        for w in word_ids
    ]
    return enc

In [31]:
# Преобразование в HF Dataset
lenta_df = Dataset.from_pandas(lenta_data, preserve_index=False).map(
    prepare_synth,
    remove_columns=['words', 'bio_labels']
)
# Объединение датасетов
full_df = concatenate_datasets([df_train, lenta_df])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [32]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)
data_collator = DataCollatorForTokenClassification(tokenizer)
tokenizer = tokenizer.from_pretrained(MODEL_CHECKPOINT)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
args = TrainingArguments(
    output_dir='ner',
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=df_train,
    eval_dataset=df_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics)

trainer.evaluate()

{'eval_loss': 2.522981643676758,
 'eval_model_preparation_time': 0.0,
 'eval_precision': 0.010755835223531133,
 'eval_recall': 0.07472359893252002,
 'eval_f1': 0.01880486751842909,
 'eval_accuracy': 0.07002704326923077,
 'eval_runtime': 10.6569,
 'eval_samples_per_second': 18.767,
 'eval_steps_per_second': 1.22}

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1,Accuracy
1,No log,1.068147,0.000000,0.000000,0.000000,0.000000,0.731641
2,No log,0.880285,0.000000,0.139342,0.077011,0.099198,0.760186
3,No log,0.784446,0.000000,0.172689,0.120854,0.142195,0.773317
4,No log,0.719371,0.000000,0.183121,0.147795,0.163572,0.784570
5,No log,0.678128,0.000000,0.199940,0.168255,0.182734,0.792803
6,No log,0.650407,0.000000,0.220036,0.200407,0.209763,0.800180
7,No log,0.632844,0.000000,0.226940,0.216673,0.221688,0.803576
8,No log,0.619003,0.000000,0.243933,0.224806,0.233979,0.808053
9,No log,0.611901,0.000000,0.250134,0.237006,0.243393,0.810291
10,0.847100,0.609754,0.000000,0.251875,0.239039,0.245289,0.810847


TrainOutput(global_step=500, training_loss=0.8470775146484375, metrics={'train_runtime': 825.9692, 'train_samples_per_second': 9.686, 'train_steps_per_second': 0.605, 'total_flos': 88880837762016.0, 'train_loss': 0.8470775146484375, 'epoch': 10.0})

In [35]:
trainer.evaluate()

{'eval_loss': 0.6097538471221924,
 'eval_model_preparation_time': 0.0,
 'eval_precision': 0.2518746652383503,
 'eval_recall': 0.23903926801372474,
 'eval_f1': 0.2452891699810915,
 'eval_accuracy': 0.8108473557692307,
 'eval_runtime': 9.1357,
 'eval_samples_per_second': 21.892,
 'eval_steps_per_second': 1.423,
 'epoch': 10.0}

In [36]:
trainer.save_model("./ner_pretrained_synthetic_model")

| Метод                    | Precision | Recall  | F1      | Accuracy |
|--------------------------|-----------|---------|---------|----------|
| Базовое дообучение       | 0.2577    | 0.2295  | 0.2428  | 0.8154   |
| MLM‑предобучение         | 0.2558    | 0.2378  | 0.2464  | 0.8138   |
| Синтетическая разметка   | 0.2519    | 0.2390  | 0.2453  | 0.8108   |

Модели имеют схожие метрики 😎, базовая модель демонстрирует наилучший precision (≈ 0.258) и самую низкую полноту (recall ≈ 0.230), дообучение с MLM слегка повышает F1 (+0.0036) за счёт роста recall, сохраняя precision на том же уровне. Синтетическая разметка даёт аналогичный прирост полноты (+0.0095 к базе), но немного уступает в precision и accuracy.

В целом, при других прогонах mlm модель показывала себя стабильнее и показывала выше метрики. Довольно странно, что ожидалось, что синтетика может сделать хуже, так как в ситнетике не присутствовали некоторые классы, из-за чего казалось что из-за дисбаланса качества модеь будет хуже, но несмотря на это метрики оказались схожи

# 4. Инференc

In [37]:
model = AutoModelForTokenClassification.from_pretrained("ner_finetuned_mlm_model") #загружаем модель
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
ner_pipe = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy='simple',)
print(ner_pipe('Россия рассчитывает на конструктивное воздействие США на Грузию\n\n04/08/2008 12:08\n\nМОСКВА, 4 авг - РИА Новости. Россия рассчитывает, что США воздействуют на Тбилиси в связи с обострением ситуации в зоне грузино-осетинского конфликта. Об этом статс-секретарь - заместитель министра иностранных дел России Григорий Карасин заявил в телефонном разговоре с заместителем госсекретаря США Дэниэлом Фридом.\n\n"С российской стороны выражена глубокая озабоченность в связи с новым витком напряженности вокруг Южной Осетии, противозаконными действиями грузинской стороны по наращиванию своих вооруженных сил в регионе, бесконтрольным строительством фортификационных сооружений", - говорится в сообщении.\n\n"Россия уже призвала Тбилиси к ответственной линии и рассчитывает также на конструктивное воздействие со стороны Вашингтона", - сообщил МИД России.'))

Device set to use cuda:0


[{'entity_group': 'GEOPOLIT', 'score': 0.29435512, 'word': 'Грузию', 'start': 57, 'end': 63}, {'entity_group': 'ORG', 'score': 0.18446374, 'word': '##ВА', 'start': 87, 'end': 89}, {'entity_group': 'PER', 'score': 0.19256142, 'word': ',', 'start': 89, 'end': 90}, {'entity_group': 'ORG', 'score': 0.1954941, 'word': 'РИА', 'start': 99, 'end': 102}, {'entity_group': 'MEDIA', 'score': 0.27083465, 'word': 'Новости.', 'start': 103, 'end': 111}, {'entity_group': 'GEOPOLIT', 'score': 0.18239816, 'word': 'Россия', 'start': 112, 'end': 118}, {'entity_group': 'GEOPOLIT', 'score': 0.22499096, 'word': 'Тбилиси', 'start': 157, 'end': 164}, {'entity_group': 'PER', 'score': 0.4409193, 'word': 'Григорий', 'start': 304, 'end': 312}, {'entity_group': 'PER', 'score': 0.61058307, 'word': 'Карасин заявил', 'start': 313, 'end': 327}, {'entity_group': 'PER', 'score': 0.5118459, 'word': '##эл', 'start': 387, 'end': 389}, {'entity_group': 'PER', 'score': 0.44602126, 'word': '##ом', 'start': 389, 'end': 391}, {'e